# 190. Agent Lightning：怎样把任意 Agent trace 切成可训练 RL transition？

> **面试问题：Agent execution 与 learner 如何解耦，event slicing、return-to-go、版本边界、失败 trace 和 credit assignment 怎样实现？**

## 先给结论

这题的关键不是调用框架，而是把状态、预算、验证器和可复放的制品合同显式化。教学代码用小数据验证不变量；生产实现仍需替换为真实模型、内核、隔离环境与线上观测。

## 一手资料

- [Agent Lightning](https://arxiv.org/abs/2508.03680)
- [RLOO](https://arxiv.org/abs/2402.14740)
- [ReAct](https://arxiv.org/abs/2210.03629)


In [ ]:
import hashlib  # 导入本单元依赖。
import json  # 导入本单元依赖。
import math  # 导入本单元依赖。
from dataclasses import asdict, dataclass  # 导入本单元依赖。
@dataclass(frozen=True)  # 计算并保存当前中间结果。
class AgentEvent:  # 定义保存状态的数据结构。
    episode: str  # 计算并保存当前中间结果。
    step: int  # 计算并保存当前中间结果。
    action: str  # 计算并保存当前中间结果。
    reward: float  # 计算并保存当前中间结果。
    policy_version: str  # 计算并保存当前中间结果。
events = [AgentEvent("e1", 0, "search", 0.0, "p1"), AgentEvent("e1", 1, "tool", 1.0, "p1")]  # 计算并保存当前中间结果。
assert len(events) == 2  # 用断言验证关键不变量。
assert events[0].episode == events[1].episode  # 用断言验证关键不变量。
assert events[0].step < events[1].step  # 用断言验证关键不变量。


## 1. 最小状态与输入合同

先说明输入输出、边界和验证 oracle；再运行下面的底层实现。


In [ ]:
def slice_episode(events):  # 定义可复用的核心函数。
    if not events or any(event.episode != events[0].episode for event in events):  # 按条件选择控制路径。
        raise ValueError("trace 必须属于同一 episode")  # 非法输入立即显式失败。
    return [(events[i], events[i + 1] if i + 1 < len(events) else None) for i in range(len(events))]  # 返回当前计算结果。
slices = slice_episode(events)  # 计算并保存当前中间结果。
assert len(slices) == 2  # 用断言验证关键不变量。
assert slices[0][1].action == "tool"  # 用断言验证关键不变量。
assert slices[-1][1] is None  # 用断言验证关键不变量。


## 2. 核心公式或状态转换

先说明输入输出、边界和验证 oracle；再运行下面的底层实现。


In [ ]:
def return_to_go(events, gamma=1.0):  # 定义可复用的核心函数。
    result = []  # 计算并保存当前中间结果。
    total = 0.0  # 计算并保存当前中间结果。
    for event in reversed(events):  # 遍历元素以累积状态。
        total = event.reward + gamma * total  # 计算并保存当前中间结果。
        result.append(total)  # 计算并保存当前中间结果。
    return list(reversed(result))  # 返回当前计算结果。
returns = return_to_go(events)  # 返回当前计算结果。
assert returns == [1.0, 1.0]  # 用断言验证关键不变量。
assert return_to_go(events, 0.5) == [0.5, 1.0]  # 用断言验证关键不变量。
assert len(returns) == len(events)  # 用断言验证关键不变量。


## 3. 候选选择与验证

先说明输入输出、边界和验证 oracle；再运行下面的底层实现。


In [ ]:
def transition_records(events, gamma=1.0):  # 定义可复用的核心函数。
    values = return_to_go(events, gamma)  # 计算并保存当前中间结果。
    return [{"action": event.action, "return": value, "version": event.policy_version} for event, value in zip(events, values)]  # 返回当前计算结果。
records = transition_records(events)  # 计算并保存当前中间结果。
assert records[0]["return"] == 1.0  # 用断言验证关键不变量。
assert records[1]["action"] == "tool"  # 用断言验证关键不变量。
assert {item["version"] for item in records} == {"p1"}  # 用断言验证关键不变量。


## 4. 主路径实现

先说明输入输出、边界和验证 oracle；再运行下面的底层实现。


In [ ]:
def validate_versions(records, expected):  # 定义可复用的核心函数。
    if any(record["version"] != expected for record in records):  # 按条件选择控制路径。
        raise ValueError("rollout 与 learner 版本不兼容")  # 非法输入立即显式失败。
    return True  # 返回当前计算结果。
assert validate_versions(records, "p1")  # 用断言验证关键不变量。
try:  # 计算并保存当前中间结果。
    validate_versions(records, "p2"); assert False  # 计算并保存当前中间结果。
except ValueError:  # 计算并保存当前中间结果。
    assert True  # 用断言验证关键不变量。


## 5. 边界与失败分支

先说明输入输出、边界和验证 oracle；再运行下面的底层实现。


In [ ]:
def credit_weight(records):  # 定义可复用的核心函数。
    total = sum(abs(record["return"]) for record in records)  # 计算并保存当前中间结果。
    return [abs(record["return"]) / max(total, 1e-9) for record in records]  # 返回当前计算结果。
weights = credit_weight(records)  # 计算并保存当前中间结果。
assert math.isclose(sum(weights), 1.0)  # 用断言验证关键不变量。
assert len(weights) == 2  # 用断言验证关键不变量。
assert all(weight >= 0 for weight in weights)  # 用断言验证关键不变量。


## 6. 正确性与基线对照

先说明输入输出、边界和验证 oracle；再运行下面的底层实现。


In [ ]:
def split_by_episode(records_by_episode):  # 定义可复用的核心函数。
    train = [episode for episode in records_by_episode if hash(episode) % 2 == 0]  # 计算并保存当前中间结果。
    test = [episode for episode in records_by_episode if hash(episode) % 2 != 0]  # 计算并保存当前中间结果。
    return train, test  # 返回当前计算结果。
train, test = split_by_episode({"e1": records, "e2": records})  # 计算并保存当前中间结果。
assert set(train).isdisjoint(test)  # 用断言验证关键不变量。
assert set(train) | set(test) == {"e1", "e2"}  # 用断言验证关键不变量。
assert len(train) + len(test) == 2  # 用断言验证关键不变量。


## 7. 成本或数据合同

先说明输入输出、边界和验证 oracle；再运行下面的底层实现。


In [ ]:
@dataclass(frozen=True)  # 计算并保存当前中间结果。
class AgentRLArtifact:  # 定义保存状态的数据结构。
    event_schema: str  # 计算并保存当前中间结果。
    return_rule: str  # 返回当前计算结果。
    version_gate: str  # 计算并保存当前中间结果。
artifact = AgentRLArtifact("event-v1", "discounted-rtg", "strict-policy-version")  # 计算并保存当前中间结果。
assert artifact.return_rule == "discounted-rtg"  # 用断言验证关键不变量。
assert "version" in artifact.version_gate  # 用断言验证关键不变量。
assert len(hashlib.sha256(json.dumps(asdict(artifact), sort_keys=True).encode()).hexdigest()) == 64  # 用断言验证关键不变量。


## 8. 制品版本与面试收束

先说明输入输出、边界和验证 oracle；再运行下面的底层实现。


In [ ]:
assert records[0]["action"] == "search"  # 用断言验证关键不变量。
assert return_to_go([AgentEvent("x", 0, "a", -1.0, "p1")]) == [-1.0]  # 用断言验证关键不变量。
assert credit_weight([{ "return": 0.0 }]) == [0.0]  # 用断言验证关键不变量。


## 面试收束

回答时按目标、状态合同、核心算法、失败分支、评测指标与发布版本组织；受控例子只证明实现不变量，不代表真实模型或生产系统性能。
